# Diabetic Retinopathy Classification

End-to-end training pipeline for the blended 2015 + 2019 fundus image datasets.
Contains the dataset, model, transforms, splitting logic, training loop, and evaluation.

Run the cells top to bottom; the last cell kicks off training.

## Configuration

Set `MODEL_TYPE` to pick the architecture. Output filenames are derived from it, so
switching here and re-running won't clobber the other model's checkpoints or plots.

In [ ]:
# "cnn"    -> FirstCNN, trained from scratch
# "resnet" -> ResNet18, pretrained weights, fine-tuned
# "resnet50" -> ResNet50, pretrained weights, fine-tuned (optional freezing of backbone)
MODEL_TYPE = "cnn"

# Set to 0 on Windows: notebook DataLoader workers use spawn there and tend to hang.
NUM_WORKERS = 2

# Input resolution. FirstCNN uses AdaptiveAvgPool so it is resolution-agnostic;
# larger inputs make small lesions (microaneurysms) more visible. The pretrained
# ResNet branch overrides this back to 224 (its ImageNet assumption).
IMG_SIZE = 320

# Loss function: "ce" = plain cross-entropy, "sord" = soft ordinal loss that gives
# partial credit for near-miss grades and penalizes distant mistakes harder.
LOSS_TYPE = "sord"
SORD_SIGMA = 1.0

## Imports and reproducibility

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    cohen_kappa_score,
)
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit

In [ ]:
SEED = 42

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
DIAGNOSIS_MAP = dict(enumerate(CLASS_NAMES))


def set_global_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


set_global_seed()

## Device check

If `torch.version.cuda` prints `None`, torch is a CPU-only build and training will
silently fall back to the CPU — reinstall from the CUDA index before running.

In [ ]:
print("torch:", torch.__version__, "| cuda build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("mps available: ", torch.backends.mps.is_available())
print("selected device:", get_device())

## Paths

Resolve the project root regardless of where the notebook is launched from.

In [ ]:
project_root = Path.cwd()
if project_root.name in {"model", "data", "notebooks"}:
    project_root = project_root.parent

DATA_ROOT = project_root / "data" / "raw" / "2019_2015_data"
CSV_PATH = DATA_ROOT / "labels" / "traintestLabels15_trainLabels19.csv"

# change this to the directory containing your resized images
# IMAGE_DIR = DATA_ROOT / "resized_traintest15_train19"
IMAGE_DIR = DATA_ROOT / "resized_ben_graham"

print(f"Project root: {project_root}")
print(f"CSV exists:   {CSV_PATH.exists()}")
print(f"Images exist: {IMAGE_DIR.exists()}")

## Dataset

In [ ]:
class BlindnessDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, id_col="image", label_col="level"):
        self.ids = df[id_col].values
        self.labels = df[label_col].values
        self.image_dir = image_dir
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_name = str(self.ids[idx])

        # Dynamically discover the file extension (.jpeg, .png, or .jpg)
        if not img_name.endswith((".png", ".jpeg", ".jpg")):
            img_path = os.path.join(self.image_dir, f"{img_name}.jpeg")
            if not os.path.exists(img_path):
                img_path = os.path.join(self.image_dir, f"{img_name}.png")
                if not os.path.exists(img_path):
                    img_path = os.path.join(self.image_dir, f"{img_name}.jpg")
        else:
            img_path = os.path.join(self.image_dir, img_name)

        try:
            image = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            raise FileNotFoundError(f"Missing image: {img_path}. Verify image_dir path.")

        image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

## Model

In [ ]:
class FirstCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution Layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)

        # Batch Normalization Layers
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)
        self.bn5 = nn.BatchNorm2d(512)

        # Pooling Layers
        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Fully Connected Layers
        self.fc1 = nn.Linear(512, 256)
        self.bn_fc1 = nn.BatchNorm1d(256)  # stabilize the dense layer
        self.dropout = nn.Dropout(0.3)  # helps prevent memory overfitting
        self.fc2 = nn.Linear(256, 5)

    def forward(self, x):
        # Conv -> BN -> ReLU -> MaxPool
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # Input 224x224 -> Outputs 112x112
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # Outputs 56x56
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # Outputs 28x28
        x = self.pool(F.relu(self.bn4(self.conv4(x))))  # Outputs 14x14
        x = self.pool(F.relu(self.bn5(self.conv5(x))))  # Outputs 7x7

        # collapse the spatial dimensions to a fixed size (1x1) for the dense layers
        x = self.adaptive_pool(x)

        # Flatten and process dense features
        x = torch.flatten(x, 1)

        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### Alternative deeper architecture (optional)

Two conv layers per block instead of one. Defines `DeepCNN` alongside `FirstCNN`;
nothing uses it unless you pass it to `train()` yourself.

In [ ]:
def make_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2),
    )


class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            make_block(3, 32),     # 224 -> 112
            make_block(32, 64),    # 112 -> 56
            make_block(64, 128),   # 56 -> 28
            make_block(128, 256),  # 28 -> 14
            make_block(256, 512),  # 14 -> 7
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 5),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

### Resnet50 pretrained and optional freeze backbone or not

In [ ]:
class ResNet50DR(nn.Module):
    def __init__(self, num_classes=5, pretrained=True, freeze_backbone=True):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        self.resnet = models.resnet50(weights=weights)

        if freeze_backbone:
            for param in self.resnet.parameters():
                param.requires_grad = False

        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.resnet(x)

## Transforms

In [ ]:
def get_transforms(img_size=None):
    img_size = img_size or IMG_SIZE

    # Fundus images have no canonical orientation, so flips and full rotation are
    # valid. hue is left untouched because color carries diagnostic signal
    # (hemorrhages, exudates). RandomResizedCrop outputs a square, which also fixes
    # the aspect-ratio squash from the old Resize((224, 224)) on non-square images.
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=180),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # Validation is deterministic: resize shortest side then center-crop to a
    # square (no squash, no augmentation).
    val_transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    return train_transform, val_transform

## Data splitting

Three things happen here:

1. A fixed demo holdout set (8 images per class) is carved out first and its whole patient families are blocked from the training pool, so demo images never leak in.
2. 2019 data is split stratified by class; 2015 data is split by patient group (both eyes of a patient stay on the same side).
3. 2015 class 0 is downsampled at the patient level to keep training tractable.

In [ ]:
def get_family_id(image_name):
    image_name = str(image_name)
    return image_name.split("_")[0] if "_" in image_name else image_name


def build_demo_holdout(df, out_path):
    """Reserve a fixed demo set before any train/val split and save it to disk."""
    demo_df = (
        df.groupby("level", group_keys=False)
        .apply(lambda group: group.sample(n=min(8, len(group)), random_state=SEED))
        .copy()
    )
    demo_df["diagnosis"] = demo_df["level"].map(DIAGNOSIS_MAP)
    demo_df = demo_df[["image", "level", "diagnosis"]].reset_index(drop=True)
    demo_df.to_csv(out_path, index=False)
    print(f"Saved demo holdout set to {out_path} ({len(demo_df)} images)")
    return demo_df


def build_splits(csv_path, val_split=0.2):
    df = pd.read_csv(csv_path)

    demo_df = build_demo_holdout(df, project_root / "data" / "demo_holdout_set.csv")
    blocked_family_ids = set(demo_df["image"].map(get_family_id))

    df["family_id"] = df["image"].map(get_family_id)
    df = df[~df["family_id"].isin(blocked_family_ids)].drop(columns=["family_id"])

    np.random.seed(SEED)
    df["is_2015"] = df["image"].apply(lambda x: "_" in str(x))
    df_2015 = df[df["is_2015"]].copy()
    df_2019 = df[~df["is_2015"]].copy()

    # Step A: Stratified split for 2019 data (perfect class balance)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_split, random_state=SEED)
    t_idx_19, v_idx_19 = next(sss.split(df_2019, df_2019["level"]))
    train_2019 = df_2019.iloc[t_idx_19].copy()
    val_2019 = df_2019.iloc[v_idx_19].copy()

    # Step B: Patient-group split for 2015 data (anti-data-leakage)
    df_2015["patient_id"] = df_2015["image"].apply(lambda x: str(x).split("_")[0])
    gss = GroupShuffleSplit(n_splits=1, test_size=val_split, random_state=SEED)
    t_idx_15, v_idx_15 = next(gss.split(df_2015, groups=df_2015["patient_id"]))
    train_2015_raw = df_2015.iloc[t_idx_15].copy()
    val_2015 = df_2015.iloc[v_idx_15].copy()

    # Step C: Downsample 2015 class 0 to cap training time
    train_2015_rare = train_2015_raw[train_2015_raw["level"] != 0]
    train_2015_zero = train_2015_raw[train_2015_raw["level"] == 0]

    target_zero_count = train_2015_rare["level"].value_counts().max()
    zero_patients = train_2015_zero["patient_id"].unique()
    sampled_zero_patients = np.random.choice(
        zero_patients,
        size=min(len(zero_patients), target_zero_count // 2),
        replace=False,
    )
    train_2015_zero_downsampled = train_2015_zero[train_2015_zero["patient_id"].isin(sampled_zero_patients)]
    train_2015_balanced = pd.concat([train_2015_rare, train_2015_zero_downsampled])

    # Step D: Combine into final clean DataFrames
    train_df = pd.concat([train_2019, train_2015_balanced], ignore_index=True).sample(frac=1, random_state=SEED)
    val_df = pd.concat([val_2019, val_2015], ignore_index=True).sample(frac=1, random_state=SEED)

    print(f"Engineered Training Set: {len(train_df)} images (downsampled for efficiency)")
    print(f"Engineered Validation Set: {len(val_df)} images (preserved for realistic testing)\n")

    return train_df, val_df

## DataLoaders

A `WeightedRandomSampler` oversamples the rare classes so each batch sees roughly balanced labels.

In [ ]:
def build_loaders(train_df, val_df, image_dir, batch_size=32, num_workers=2, img_size=None):
    train_transform, val_transform = get_transforms(img_size=img_size)

    # Weight every sample by the inverse frequency of its class
    train_labels = train_df["level"].values
    class_counts = np.bincount(train_labels)
    class_weights = 1.0 / class_counts
    sample_weights = torch.from_numpy(class_weights[train_labels]).double()

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=torch.Generator().manual_seed(SEED),
    )

    train_set = BlindnessDataset(train_df, image_dir, transform=train_transform, id_col="image", label_col="level")
    val_set = BlindnessDataset(val_df, image_dir, transform=val_transform, id_col="image", label_col="level")

    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED),
    )
    val_loader = DataLoader(
        val_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED),
    )

    return train_loader, val_loader

## Train and evaluate steps

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(loader, desc="  Train", leave=False)
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += images.size(0)
        loop.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct / total:.3f}")

    return total_loss / total, correct / total

In [ ]:
def _safe_qwk(labels, preds):
    # QWK is undefined when only one class is present in a slice.
    if len(set(labels)) < 2:
        return float("nan")
    return cohen_kappa_score(labels, preds, weights="quadratic", labels=[0, 1, 2, 3, 4])


def evaluate(model, loader, criterion, device, show_report=True, show_plot=False, plot_title=None,
             plot_path="confusion_matrix.png", source_flags=None):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        loop = tqdm(loader, desc="  Val  ", leave=False)
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            loop.set_postfix(loss=f"{loss.item():.4f}")

    total = len(all_labels)
    correct = sum(p == l for p, l in zip(all_preds, all_labels))

    if show_report:
        macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        qwk = _safe_qwk(all_labels, all_preds)
        print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))
        print(f"Macro F1: {macro_f1:.4f}  |  Quadratic Weighted Kappa: {qwk:.4f}")
        print("Confusion matrix:")
        print(confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3, 4]))

        # Break metrics down by data source to test whether the model is keying off
        # 2015-vs-2019 artifacts rather than disease (e.g. No-DR -> Proliferative errors).
        if source_flags is not None:
            flags = np.asarray(source_flags, dtype=bool)
            preds_arr = np.asarray(all_preds)
            labels_arr = np.asarray(all_labels)
            if len(flags) != len(labels_arr):
                print(f"\n[warn] source_flags length {len(flags)} != {len(labels_arr)}; "
                      f"skipping source breakdown.")
            else:
                for name, mask in [("2015 (crowd-graded)", flags), ("2019 (expert-graded)", ~flags)]:
                    n = int(mask.sum())
                    if n == 0:
                        continue
                    sub_labels = labels_arr[mask].tolist()
                    sub_preds = preds_arr[mask].tolist()
                    src_f1 = f1_score(sub_labels, sub_preds, average="macro", zero_division=0)
                    src_qwk = _safe_qwk(sub_labels, sub_preds)
                    print(f"\n--- Source: {name}  (n={n})  "
                          f"macro F1={src_f1:.4f}  QWK={src_qwk:.4f} ---")
                    print(classification_report(
                        sub_labels, sub_preds, labels=[0, 1, 2, 3, 4],
                        target_names=CLASS_NAMES, zero_division=0,
                    ))

    if show_plot:
        cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3, 4])
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
        disp.plot(cmap="Blues", values_format="d")
        plt.title(plot_title or "Validation Confusion Matrix")
        plt.tight_layout()
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.show()

    return total_loss / total, correct / total

In [ ]:
def plot_loss_curves(train_losses, val_losses, plot_path="loss_curves.png"):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_losses, label="Training Loss")
    plt.plot(epochs, val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss Curves")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Soft ordinal loss (SORD). DR severity is ordinal (0..4), so instead of a hard
# one-hot target we spread probability mass onto neighbouring grades, weighted by
# squared ordinal distance. This gives partial credit for near-miss predictions and
# penalizes distant mistakes (e.g. No-DR -> Proliferative) far more than plain CE.
class SORDLoss(nn.Module):
    def __init__(self, num_classes, sigma=1.0):
        super().__init__()
        self.sigma = sigma
        ranks = torch.arange(num_classes, dtype=torch.float)
        # dist[y, k] = (k - y)^2
        self.register_buffer("dist", (ranks[None, :] - ranks[:, None]) ** 2)

    def forward(self, logits, target):
        soft_target = F.softmax(-self.dist[target] / (2 * self.sigma ** 2), dim=1)
        log_prob = F.log_softmax(logits, dim=1)
        return -(soft_target * log_prob).sum(dim=1).mean()


def make_criterion(loss_type, num_classes, device, sigma=SORD_SIGMA):
    if loss_type == "ce":
        return nn.CrossEntropyLoss()
    if loss_type == "sord":
        return SORDLoss(num_classes, sigma=sigma).to(device)
    raise ValueError(f"Unknown LOSS_TYPE: {loss_type!r}. Use 'ce' or 'sord'.")

## Training loop

The lowest validation loss across epochs is checkpointed to `<run_name>_best_model.pth`;
the final epoch's weights go to `<run_name>_model.pth`. `run_name` defaults to `MODEL_TYPE`.

In [ ]:
def train(
    model,
    run_name=None,
    csv_path=None,
    image_dir=None,
    num_epochs=25,
    batch_size=32,
    lr=3e-4,
    val_split=0.2,
    num_workers=None,
    img_size=None,
    loss_type=None,
):
    run_name = run_name or MODEL_TYPE
    csv_path = csv_path or CSV_PATH
    image_dir = image_dir or IMAGE_DIR
    num_workers = NUM_WORKERS if num_workers is None else num_workers
    img_size = img_size or IMG_SIZE
    loss_type = loss_type or LOSS_TYPE

    best_path = f"{run_name}_best_model.pth"
    final_path = f"{run_name}_model.pth"

    set_global_seed()
    device = get_device()
    print(f"Using device: {device} | img_size: {img_size} | loss: {loss_type}\n")

    train_df, val_df = build_splits(csv_path, val_split=val_split)
    val_source = val_df["is_2015"].values
    train_loader, val_loader = build_loaders(
        train_df, val_df, image_dir, batch_size=batch_size, num_workers=num_workers, img_size=img_size
    )

    model = model.to(device)
    # optimize unfrozen parameters only (e.g. the final classifier layer of a pretrained ResNet)
    optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = make_criterion(loss_type, num_classes=len(CLASS_NAMES), device=device)

    best_val_loss = float("inf")
    best_epoch = 0
    train_losses = []
    val_losses = []

    epoch_bar = tqdm(range(1, num_epochs + 1), desc="Epochs")
    for epoch in epoch_bar:
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device, show_report=False)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        epoch_bar.write(
            f"Epoch {epoch:>2}/{num_epochs} | "
            f"train loss {train_loss:.4f}  acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f}  acc {val_acc:.3f}"
        )
        epoch_bar.set_postfix(val_loss=f"{val_loss:.4f}", val_acc=f"{val_acc:.3f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            torch.save(model.state_dict(), best_path)
            epoch_bar.write(
                f"--> Found better weights! Saved checkpoint to {best_path} (val loss: {best_val_loss:.4f})"
            )

    plot_loss_curves(train_losses, val_losses, plot_path=f"{run_name}_loss_curves.png")

    torch.save(model.state_dict(), final_path)
    print(f"Final epoch model saved to {final_path}")

    model.load_state_dict(torch.load(best_path, map_location=device))
    print(f"Loaded {best_path} back into the model for final evaluation (best epoch: {best_epoch}).")
    evaluate(
        model,
        val_loader,
        criterion,
        device,
        show_plot=True,
        plot_title=f"Validation Confusion Matrix - Best Epoch {best_epoch}",
        plot_path=f"{run_name}_confusion_matrix.png",
        source_flags=val_source,
    )

    print(f"Training complete. Best validation loss achieved: {best_val_loss:.4f}")
    return model, {"train_losses": train_losses, "val_losses": val_losses, "best_epoch": best_epoch}

## Run training

Which model runs is decided by `MODEL_TYPE` in the config cell at the top.

In [ ]:
if MODEL_TYPE == "cnn":
    net = FirstCNN()
    num_epochs, lr, img_size = 25, 3e-4, IMG_SIZE
elif MODEL_TYPE == "resnet":
    net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    net.fc = nn.Linear(net.fc.in_features, 5)
    # Pretrained ResNet expects ImageNet-style 224 inputs.
    num_epochs, lr, img_size = 10, 1e-4, 224
elif MODEL_TYPE == "resnet50":
    net = ResNet50DR(num_classes=len(CLASS_NAMES), pretrained=True, freeze_backbone=True)
    # can try 3e-4 or 1-e4
    num_epochs, lr, img_size = 10, 3e-4, 224
else:
    raise ValueError(f"Unknown MODEL_TYPE: {MODEL_TYPE!r}. Use 'cnn' or 'resnet'.")

print(f"Training MODEL_TYPE={MODEL_TYPE} for {num_epochs} epochs at lr={lr}, img_size={img_size}\n")
model, history = train(net, num_epochs=num_epochs, lr=lr, img_size=img_size)